In [2]:
import data.breathe_data as bd
import data.helpers as dh
import src.models.helpers as mh
from plotly.subplots import make_subplots
import datetime
import data.cfr_data_19_23 as cfrd
import pandas as pd
import cfr.cfr_viz_helpers as vh
import plotly.express as px
import plotly.graph_objs as go

In [2]:
# Plot predictions - baseline against associated variables
# Ensure completeness on 2023 baseline data, 2023 and 2019 predictions, associations

In [3]:
# Load baseline data
df_meas = bd.load_meas_from_excel(
    "CF_Registry_19_23_processed_with_idx", study_folder="CFR"
)
df_meas.columns

Index(['ID', 'Age', 'Height', 'FEV1', 'FEF2575', 'Sex', 'Date Recorded',
       'ecFEV1', 'ecFEF2575', 'ecFEF2575%ecFEV1', 'Predicted FEV1',
       'ecFEV1 % Predicted', 'FEV1 % Predicted', 'idx ecFEV1 (L)',
       'idx ecFEF25-75 % ecFEV1 (%)'],
      dtype='object')

In [4]:
# Load AC predictions
df_res = bd.load_meas_from_excel(
    # "infer_AR_using_19_23_data_2entries_fev1_10122025",
    "infer_AR_using_19_23_data_2entries_fev1_fef2575_10122025",
    study_folder="CFR",
    str_cols_to_arrays=["Airway resistance (%)"],
).drop(columns="Healthy FEV1 (L)")
df_res23 = df_res[df_res["Date Recorded"] == datetime.date(2023, 1, 1)]
print(f"Shape: {df_res23.shape}")

Shape: (1485, 3)


In [5]:
# Merge predictions and baseline data
meascols = ["ID", "Date Recorded", "ecFEV1 % Predicted"]
df = df_res23.merge(df_meas[meascols], on=["ID", "Date Recorded"])
print(f"Shape dfmeas + df_res23: {df.shape}")
# CCL: Results are complete which is expected because computed based on df_meas

Shape dfmeas + df_res23: (1485, 4)


In [6]:
# Load IV data from 2019-23
df_ass = pd.read_excel(dh.get_path_to_main() + "ExcelFiles/CFR/IV_data_19-23.xlsx")

df_avg_ivs_per_year = (
    df_ass.groupby("ID")
    .agg({"Home IVs": "mean", "Hosp IVs": "mean"})
    .rename(columns={"Home IVs": "Avg Home IVs", "Hosp IVs": "Avg Hosp IVs"})
)

In [7]:
# Merge associated variables into the df
df = df.merge(df_avg_ivs_per_year.reset_index(), on="ID")
print(f"Shape final df: {df.shape}")
# Associations are complete

Shape final df: (1485, 6)


In [8]:
df.head()

,ID,Date Recorded,Airway resistance (%),ecFEV1 % Predicted,Avg Home IVs,Avg Hosp IVs
0,B155916,2023-01-01,"[2.84510378e-06, 1.19407844e-05, 4.56567456e-0...",60.003459,0.2,0.4
1,B155917,2023-01-01,"[0.00189416089, 0.00577542141, 0.0159695991, 0...",86.371404,0.0,0.0
2,B155918,2023-01-01,"[0.132413403, 0.153014939, 0.165978846, 0.1513...",108.395228,0.6,0.2
3,B155921,2023-01-01,"[2.85703336e-07, 1.0518237e-06, 3.66306036e-06...",53.026292,0.2,0.2
4,B155925,2023-01-01,"[5.83818829e-10, 2.31597484e-09, 8.44808723e-0...",34.396399,1.8,2.8


In [ ]:
AR = mh.VariableNode("Airway resistance (%)", 0, 90, 2, {"type": "uniform"})
AC = mh.VariableNode("Airway conductance (%)", 10, 100, 2, {"type": "uniform"})

df[AC.name] = df[AR.name].apply(lambda arr: arr[::-1])

df["mean AC"] = df[AC.name].apply(lambda ac: AC.get_mean(ac))

# diff = Predicted AC - baseline FEV1%pred
df["diff"] = df["mean AC"] - df["ecFEV1 % Predicted"]

df["clipped ecFEV1%Predicted"] = df["ecFEV1 % Predicted"].clip(upper=100)
df["diff (clipped)"] = df["mean AC"] - df["clipped ecFEV1%Predicted"]

In [85]:
df["AC std"] = df[AC.name].apply(lambda ac: AC.get_std(ac))
df["Large diff"] = abs(df["diff (clipped)"]) > df["AC std"]
df["Below 100%"] = df["ecFEV1 % Predicted"] < 100

In [89]:
import plotly.express as px

# Scatter plot with marginal distribution (y axis) for Avg Home IVs
xcol = ""
xcol = " (clipped)"

df_plot = df
# df_plot = df[df["Large diff"]]
# df_plot = df[df["Below 100%"]]

for col in ["Hosp", "Home"]:
    title = f"Association of model output diff against baseline with {col} IVs (2019-23) - large diff"
    fig = px.scatter(
        df_plot,
        x=f"diff{xcol}",
        y=f"Avg {col} IVs",
        labels={
            f"diff{xcol}": f"Predicted conductance - Baseline ppFEV1{xcol}",
            f"Avg {col} IVs": f"Average number of {col} IVs",
        },
        # marginal_y="histogram",
        title=title,
        size_max=6,  # controls the maximum bubble size
        size=[3] * len(df_plot),
        hover_data=["ID"],  # Add this line to include df.ID in hover label
    )
    # fig.update_traces(marker=dict(size=3))
    fig.update_layout(height=600, width=800)
    fig.show()
    # fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/{title}.pdf")

In [ ]:
# Why so many individuals with worse lungs have no hosp IVs?

# Home and hops IVs have a similar pattern

# Inaccuracies
# 1. Deal with individuals that have FEV1% > 100%

# Biases
# More people with negative diff
# 2. Few individuals have 3+ IVs

# TODO
# 1. Show marked difference by excluding IDs when baseline is within 1 sigma from prediction
# 2. Compute percentage of people per number of IVs?
# Verify the pattern on other years
# Plot home/hosp IVs between 2019 and 2023. Max 20 per year. Check IV data completeness over the years, if missing values then compute an average number

In [1]:
# Scatter plot with marginal distribution (y axis) for Avg Home IVs
xcol = ""
xcol = " (clipped)"

df_plot = df
# df_plot = df[df["Large diff"]]
# df_plot = df[df["Below 100%"]]

# Create the three dataframes
df_mild = df_plot[df_plot['mean AC'] >= 70]
df_moderate = df_plot[(df_plot["mean AC"] >= 40) & (df_plot["mean AC"] < 70)]
df_severe = df_plot[(df_plot["mean AC"] < 40)]

for col in ["Hosp"]:#, "Home"]:
    title = f"Association of model output diff against baseline with {col} IVs (2019-23) - large diff"
    fig = make_subplots(rows=1, cols=3)

    for i, dftmp in enumerate([df_mild, df_moderate, df_severe]):
        fig.add_trace(
            go.Scatter(
                x=dftmp[f"diff{xcol}"],
                y=dftmp[f"Avg {col} IVs"],
                mode="markers",
                marker=dict(color="#0072b2", size=3)
            ),
            row=1,
            col=i+1,
        )

    # fig.update_traces(marker=dict(size=3))
    fig.update_layout(height=500, width=1000)
    fig.show()
    # fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/{title}.pdf")

NameError: name 'df' is not defined

In [ ]:
from scipy.stats import pearsonr

# Compute correlation between 'ecFEV1 % Predicted' and 'Avg Hosp IVs'
x1 = df["ecFEV1 % Predicted"]
y1 = df["Avg Hosp IVs"]
corr1, pval1 = pearsonr(x1, y1)

# Compute correlation between 'mean AC' and 'Avg Hosp IVs'
x2 = df["mean AC"]
y2 = df["Avg Hosp IVs"]
corr2, pval2 = pearsonr(x2, y2)

Pearson r between ecFEV1 % Predicted and Avg Hosp IVs: -0.392 (p=1.37e-55); Increase is significant at alpha=0.05
Pearson r between mean AC and Avg Hosp IVs: -0.394 (p=2.18e-56); Increase is significant at alpha=0.05


In [119]:
import plotly.express as px
import numpy as np

# Scatter plot with marginal distribution (y axis) for Avg Home IVs
xcol = ""
xcol = " (clipped)"

df_plot = df
# df_plot = df[df["Large diff"]]
# df_plot = df[df["Below 100%"]]

for col in ["Hosp", "Home"]:
    title = f"Association of model output diff against baseline with {col} IVs (2019-23) - large diff"
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_plot['clipped ecFEV1%Predicted'],
        y=df_plot[f"Avg {col} IVs"],
        mode="markers"
    ))
    fig.add_trace(go.Scatter(
        x=df_plot['mean AC'],
        y=df_plot[f"Avg {col} IVs"],
        mode="markers"
    ))

    fig.update_traces(marker=dict(size=3))
    fig.update_layout(height=800, width=800)
    fig.show()
    # fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/{title}.pdf")